## 在Macbook上私有化模型部署，ChatGLM2-6B-int4
**1.ChatGLM2-6B项目仓库：**
   - https://github.com/THUDM/ChatGLM2-6B

**2.chatglm.cpp项目仓库：**
   - https://github.com/li-plus/chatglm.cpp
    *因为要实现 Mac 笔记本上实时对话，所以选chatglm.cpp，chatglm.cpp 类似 llama.cpp 的 CPU 量化加速推理方案*

**3.Models：模型选择chatglm2-6b-int4**
   - huggingface社区下载模型地址:
   - https://huggingface.co/THUDM/chatglm2-6b-int4/tree/main
** **
![mac_chatglm推理](../images/chatglm_mac.png)
<br />

** **
### 具体步骤参见 https://github.com/li-plus/chatglm.cpp，以及 https://huggingface.co/THUDM/chatglm2-6b-int4

<br />

**1.Clone the ChatGLM.cpp repository into your local machine**

<br />

```shell
    git clone --recursive https://github.com/li-plus/chatglm.cpp.git && cd chatglm.cpp
    git submodule update --init --recursive
```
<br />

**2.参照huggingface社区下载CHATGLM模型，（这里我选择的model：chatglm2-6b-int4）**

<br />

```shell
    python -m pip install -U pip
    pip install --upgrade torch
    pip install protobuf transformers==4.30.2 cpm_kernels \ gradio mdtex2html sentencepiece accelerate

    进入python命令行，下载chatglm2-6b-int4模型文件，忽略下载之外的错误：
    from transformers import AutoTokenizer, AutoModel
    tokenizer = AutoTokenizer.from_pretrained("THUDM/chatglm2-6b-int4", trust_remote_code=True)
    model = AutoModel.from_pretrained("THUDM/chatglm2-6b-int4", trust_remote_code=True).half().cuda()
    

    进入chatglm.cpp工程，执行如下命令：
    python3 chatglm_cpp/convert.py -i THUDM/chatglm2-6b-int4 -t q4_0 -o chatglm-ggml.bin

    brew install cmake
    cmake -B build
    cmake --build build -j --config Release

    测试：
    ./build/bin/main -m chatglm-ggml.bin -p 你好
    
```
<br />

**3.启动LangChain API --  Start the api server for LangChain:**

<br />

```shell
    pip install 'chatglm-cpp[api]'
    
    cd chatglm.cpp/chatglm_cpp
    MODEL=../chatglm-ggml.bin uvicorn chatglm_cpp.langchain_api:app --host 127.0.0.1 --port 8000

    新开窗口，测试：
    curl http://127.0.0.1:8000 -H 'Content-Type: application/json' -d '{"prompt": "你好"}'
```
** **


In [8]:
# 直接调模型chatglm_cpp
import chatglm_cpp

chatglm_ggml = "/Users/mac/Downloads/lesson/chatglm.cpp/chatglm-ggml.bin"  # 替换成自己本地的模型路径
pipeline = chatglm_cpp.Pipeline(chatglm_ggml)
history = [chatglm_cpp.ChatMessage(role="user", content="你好, 你是我的英中翻译助手：小智，请帮我将下面的英文内容翻译成中文 \n I am interested in Ai and have been studying it recently")]
res = pipeline.chat(history)
history.append(res)
print(res)
print(history)


ChatMessage(role="assistant", content="我对人工智能(AI)感兴趣，并且最近一直在学习它。", tool_calls=[])
[ChatMessage(role="user", content="你好, 你是我的英中翻译助手：小智，请帮我将下面的英文内容翻译成中文 
 I am interested in Ai and have been studying it recently", tool_calls=[]), ChatMessage(role="assistant", content="我对人工智能(AI)感兴趣，并且最近一直在学习它。", tool_calls=[])]


In [9]:
history.append(chatglm_cpp.ChatMessage(role="user", content="你叫什么名字？"))
res = pipeline.chat(history)
print(res)

ChatMessage(role="assistant", content="我是一个人工智能助手，你可以直接称呼我为小智。", tool_calls=[])


In [10]:
# 使用Langchain调用chatglm_cpp模型
from langchain.llms import ChatGLM
from langchain.schema import SystemMessage, HumanMessage

endpoint_url = (
    "http://127.0.0.1:8000"
)
history = []
chatglm_llm = ChatGLM(
    endpoint_url=endpoint_url,
    history=history
)

In [11]:
messages = [
    SystemMessage(content="you are my translator assisstant and your name is Jack, translate following english into chinese. "),
    HumanMessage(content="I am a lovely girl")
]

In [12]:
result = chatglm_llm.predict_messages(messages)
messages.append(result)
result

/opt/miniconda3/envs/langchain/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `BaseLLM.predict_messages` was deprecated in langchain-core 0.1.7 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


AIMessage(content='人类：我是个可爱的女孩')

In [13]:
messages.append(HumanMessage(content="My favorite basketball player is Kobe"))
result = chatglm_llm.predict_messages(messages)
messages.append(result)
result

AIMessage(content='AI: 我最喜欢的篮球运动员是科比')

In [14]:
print(messages)

[SystemMessage(content='you are my translator assisstant and your name is Jack, translate following english into chinese. '), HumanMessage(content='I am a lovely girl'), AIMessage(content='人类：我是个可爱的女孩'), HumanMessage(content='My favorite basketball player is Kobe'), AIMessage(content='AI: 我最喜欢的篮球运动员是科比')]


In [15]:
messages.append(HumanMessage(content="你的名字是什么"))
result_m = chatglm_llm.predict_messages(messages)
result_m

AIMessage(content='AI: 你的名字是杰克')

### 基于ChatGLM大语言模型的LLMChain

In [18]:
from langchain import LLMChain, PromptTemplate

prompt_template = "Tell me a {adjective} joke"
prompt = PromptTemplate(
    input_variables=["adjective"], template=prompt_template
)
llm = LLMChain(llm=chatglm_llm, prompt=prompt, verbose=True)
llm.run({"adjective": "cold"})




> Entering new LLMChain chain...
Prompt after formatting:
Tell me a cold joke

> Finished chain.


"Sure, here's a classic cold joke:\n\nWhy don't scientists trust atoms?\n\nBecause they make up everything!"

In [19]:
llm.run({"adjective": "math"})




> Entering new LLMChain chain...
Prompt after formatting:
Tell me a math joke

> Finished chain.


"Sure, here's one:\n\nWhy did the math book look so sad?\n\nBecause it had too many problems."

### 新增translation_chain.py中关于ChatGLM的代码，读取配置为config.yaml中的model_name: "chat_glm"

### OpenAI-Translator 2.0 优化版的改动说明
在 openai-translator gradio 图形化界面基础上，支持风格化翻译，如：小说、新闻稿、作家风格等。
修改gradio_server、flask_server、pdf_translator，增加translation_style参数，修改translation_chain的prompt以接收这个参数。修改writer.py增加 _风格名作为后缀。